# Modelado de datos

In [1]:
%load_ext watermark
%watermark

Last updated: 2022-10-12T00:15:22.492427-05:00

Python implementation: CPython
Python version       : 3.7.4
IPython version      : 7.8.0

Compiler    : MSC v.1915 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 158 Stepping 9, GenuineIntel
CPU cores   : 8
Architecture: 64bit



In [2]:
import pandas as pd
%matplotlib inline 
# no crea widget ni nada para no presentar resultados.
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
import matplotlib as plt
import seaborn as sns
from matplotlib.pyplot import figure
import warnings
warnings.filterwarnings("ignore")


plt.rcParams['figure.figsize'] = (10,10) # 10 pulgadas de ancho y 10 de alto para todos los plots

In D:\ProgramData\Anaconda3\envs\EnvDataLab\lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test.mplstyle: 
The text.latex.preview rcparam was deprecated in Matplotlib 3.3 and will be removed two minor releases later.
In D:\ProgramData\Anaconda3\envs\EnvDataLab\lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test.mplstyle: 
The mathtext.fallback_to_cm rcparam was deprecated in Matplotlib 3.3 and will be removed two minor releases later.
In D:\ProgramData\Anaconda3\envs\EnvDataLab\lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test.mplstyle: Support for setting the 'mathtext.fallback_to_cm' rcParam is deprecated since 3.3 and will be removed two minor releases later; use 'mathtext.fallback : 'cm' instead.
In D:\ProgramData\Anaconda3\envs\EnvDataLab\lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test.mplstyle: 
The validate_bool_maybe_none function was deprecated in Matplotlib 3.3 and will be removed two minor releases later.
In D:\ProgramData\Anaconda3\

In [10]:
bank_dp = pd.read_parquet("../output/data-bank_4_PreModeling.parquet",engine='pyarrow')

In [11]:
bank_dp.head(5)

,edad,flag_mora,prom_saldoAnual,flag_hipotecario,flag_prestamo,tipo_contacto,dia_UltContacto,seg_UltContacto,cant_contactoCampanha,cant_contactoBeforeCampanha,...,mes_UltContacto_nov,mes_UltContacto_oct,mes_UltContacto_sep,estadoCivil_divorced,estadoCivil_married,estadoCivil_single,educacion_primary,educacion_secondary,educacion_tertiary,educacion_unknown
0,59,1,2343,1,0,2,5,1042,1,0,...,0,0,0,0,1,0,0,1,0,0
1,56,1,45,0,0,2,5,1467,1,0,...,0,0,0,0,1,0,0,1,0,0
2,41,1,1270,1,0,2,5,1389,1,0,...,0,0,0,0,1,0,0,1,0,0
3,55,1,2476,1,0,2,5,579,1,0,...,0,0,0,0,1,0,0,1,0,0
4,54,1,184,0,0,2,5,673,2,0,...,0,0,0,0,1,0,0,0,1,0


In [12]:
'''Check numerical data set'''
display(bank_dp.head(3), bank_dp.shape)
display('Observamos 11152 registros y 73 columnas numéricas luego de la transformación.')

,edad,flag_mora,prom_saldoAnual,flag_hipotecario,flag_prestamo,tipo_contacto,dia_UltContacto,seg_UltContacto,cant_contactoCampanha,cant_contactoBeforeCampanha,...,mes_UltContacto_nov,mes_UltContacto_oct,mes_UltContacto_sep,estadoCivil_divorced,estadoCivil_married,estadoCivil_single,educacion_primary,educacion_secondary,educacion_tertiary,educacion_unknown
0,59,1,2343,1,0,2,5,1042,1,0,...,0,0,0,0,1,0,0,1,0,0
1,56,1,45,0,0,2,5,1467,1,0,...,0,0,0,0,1,0,0,1,0,0
2,41,1,1270,1,0,2,5,1389,1,0,...,0,0,0,0,1,0,0,1,0,0


(11152, 45)

'Observamos 11152 registros y 73 columnas numéricas luego de la transformación.'

## 1. Selecionar el resultado de la métrica

Se propone utilizar métricas ROC_AUC para evaluar diferentes modelos con monitoreo adicional de la dinámica de la métrica de precisión. 

Este enfoque nos permitirá explorar modelos desde diferentes ángulos.

## 2. Separar data de entrenamiento y test

In [7]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
import gc 

In [8]:
train_test_split?

### Muestreo Estratificado

Es un concepto importante que a menudo se pasa por alto al desarrollar un modelo, ya sea para regresión o clasificación. Recuerde, que para evitar el sobreajuste de nuestros datos, debemos implementar una validación cruzada, sin embargo, debemos asegurarnos de que al menos las características que tienen la mayor influencia en nuestra etiqueta (si un cliente potencial abrirá un depósito a plazo o no) es igualmente distribuido. ¿Qué quiero decir con esto?

In [16]:
bank_dp.result_campanha.unique()

array([0, 1], dtype=uint8)

In [13]:
bank_dp.columns

Index(['edad', 'flag_mora', 'prom_saldoAnual', 'flag_hipotecario',
       'flag_prestamo', 'tipo_contacto', 'dia_UltContacto', 'seg_UltContacto',
       'cant_contactoCampanha', 'cant_contactoBeforeCampanha',
       'result_campanha', 'FlagTarget_DP', 'Rango_edad', 'duration_status',
       'trabajo_admin.', 'trabajo_blue-collar', 'trabajo_entrepreneur',
       'trabajo_housemaid', 'trabajo_management', 'trabajo_retired',
       'trabajo_self-employed', 'trabajo_services', 'trabajo_student',
       'trabajo_technician', 'trabajo_unemployed', 'trabajo_unknown',
       'mes_UltContacto_apr', 'mes_UltContacto_aug', 'mes_UltContacto_dec',
       'mes_UltContacto_feb', 'mes_UltContacto_jan', 'mes_UltContacto_jul',
       'mes_UltContacto_jun', 'mes_UltContacto_mar', 'mes_UltContacto_may',
       'mes_UltContacto_nov', 'mes_UltContacto_oct', 'mes_UltContacto_sep',
       'estadoCivil_divorced', 'estadoCivil_married', 'estadoCivil_single',
       'educacion_primary', 'educacion_secondary', 'e

Por ejemplo para el préstamo personal `flag_prestamo`:

Tener un préstamo personal es una característica importante que determina si un cliente potencial abrirá un depósito a plazo o no

Pasos para implementar muestreo estratificado:

- 1) Necesitamos ver cómo se distribuyen nuestros datos. 
- 2) Después de notar que la columna de préstamo contiene 87% de "no" (No tiene préstamos personales) y 13% de "sí" (Tiene préstamos personales). 
- 3) Queremos asegurarnos de que nuestro conjunto de entrenamiento y prueba contenga la misma proporción de 87% "no" y 13% "sí".

Distribuciión de `flag_prestamo`

In [14]:
bank_dp["flag_prestamo"].value_counts()/len(bank_dp)

0    0.86944
1    0.13056
Name: flag_prestamo, dtype: float64

In [15]:
X = bank_dp.drop(columns = 'FlagTarget_DP')
y = bank_dp[['FlagTarget_DP']]

In [29]:
gc?

In [18]:
''' Split data on train and test'''
# set global random state
random_state = 42
# split data
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state ,shuffle=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)
# collect excess data
gc.collect()

11

In [32]:
train_test_split?

Comprobar el muestreo estratificado

In [20]:
X_train["flag_prestamo"].value_counts()/len(X_train)

0    0.870194
1    0.129806
Name: flag_prestamo, dtype: float64

In [21]:
y_train["FlagTarget_DP"].value_counts()/len(y_train)

0    0.525838
1    0.474162
Name: FlagTarget_DP, dtype: float64

In [48]:
X_test["flag_prestamo"].value_counts()/len(X_test)

0    0.866428
1    0.133572
Name: flag_prestamo, dtype: float64

In [22]:
y_test["FlagTarget_DP"].value_counts()/len(y_test)

0    0.526221
1    0.473779
Name: FlagTarget_DP, dtype: float64

Si se compara la distribución del dataset para (x,y) train como para test las distribuciones son similares, de esta manera vemos que los datos una distribución similarmente alta.

In [23]:
display('El tamaño de los datos sepadaso en train y test son:', X_train.shape, y_train.shape, X_test.shape, y_test.shape)

'El tamaño de los datos sepadaso en train y test son:'

(8921, 44)

(8921, 1)

(2231, 44)

(2231, 1)

## 3. Modelado

### 3.1. Nota sobre el algoritmo de Árbol de Decisión

Para el árbol de decisión se emplea el algoritmo **CART** (*Classification and Regression Trees*), que es la implementación disponible en `DecisionTreeClassifier` de scikit-learn. Se documenta así explícitamente para mantener la coherencia entre el objetivo, la implementación y los resultados.

Todos los modelos se entrenan con `random_state=42` para garantizar la reproducibilidad. Las variables auxiliares `Rango_edad` y `duration_status`, creadas únicamente para el análisis exploratorio (EDA), se excluyen del modelado por ser derivadas de `edad` y `seg_UltContacto`, que ya están presentes.

Se eliminan las variables auxiliares del EDA antes de entrenar.

In [ ]:
# Eliminar variables auxiliares del EDA (derivadas, solo para análisis)
cols_aux = [c for c in ['Rango_edad','duration_status'] if c in bank_dp.columns]
bank_dp = bank_dp.drop(columns=cols_aux)
print('Columnas auxiliares eliminadas:', cols_aux)
print('Shape para modelado:', bank_dp.shape)

Columnas auxiliares eliminadas: ['Rango_edad', 'duration_status']
Shape para modelado: (11152, 43)


Partición estratificada 80/20 con semilla fija.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
X = bank_dp.drop(columns='FlagTarget_DP')
y = bank_dp['FlagTarget_DP']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)
print('Train:', X_train.shape, '| Test:', X_test.shape)

Train: (8921, 42) | Test: (2231, 42)


Escalado estándar para los modelos sensibles a la escala (Regresión Logística y SVM).

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_sc = scaler.transform(X_train)
X_test_sc  = scaler.transform(X_test)
X_sc       = scaler.transform(X)   # para validación cruzada

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

### 3.2. Definición de los cuatro modelos

In [ ]:
modelos = {
    'Árbol de Decisión (CART)': (DecisionTreeClassifier(random_state=RANDOM_STATE), False),
    'Regresión Logística':      (LogisticRegression(max_iter=1000, random_state=RANDOM_STATE), True),
    'Naïve Bayes':              (GaussianNB(), False),
    'SVM':                      (SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE), True),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
resultados = []
for nombre, (modelo, escalar) in modelos.items():
    Xtr = X_train_sc if escalar else X_train.values
    Xte = X_test_sc  if escalar else X_test.values
    Xcv = X_sc       if escalar else X.values
    modelo.fit(Xtr, y_train)
    y_pred  = modelo.predict(Xte)
    y_proba = modelo.predict_proba(Xte)[:, 1]
    acc_train = accuracy_score(y_train, modelo.predict(Xtr))
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_proba)
    cm   = confusion_matrix(y_test, y_pred)
    cvs  = cross_val_score(modelo, Xcv, y, cv=cv, scoring='accuracy')
    resultados.append({'modelo':nombre,'accuracy':acc,'precision':prec,'recall':rec,
                       'f1':f1,'auc':auc,'accuracy_train':acc_train,
                       'cv_mean':cvs.mean(),'cv_std':cvs.std()})
    print(f'\n=== {nombre} ===')
    print(f'  Entrenamiento (accuracy): {acc_train:.4f}')
    print(f'  Prueba  -> accuracy={acc:.4f}  precision={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}  auc={auc:.4f}')
    print(f'  Validación cruzada (5) -> media={cvs.mean():.4f}  desv.est.={cvs.std():.4f}')
    print(f'  Matriz de confusión: {cm.tolist()}')


=== Árbol de Decisión (CART) ===
  Entrenamiento (accuracy): 1.0000   <-- sobreajuste (overfitting)
  Prueba  -> accuracy=0.7853  precision=0.7806  recall=0.7606  f1=0.7705  auc=0.7841
  Validación cruzada (5) -> media=0.7833  desv.est.=0.0104
  Matriz de confusión: [[948, 226], [253, 804]]

=== Regresión Logística ===
  Entrenamiento (accuracy): 0.8213
  Prueba  -> accuracy=0.8203  precision=0.8280  recall=0.7833  f1=0.8051  auc=0.9071
  Validación cruzada (5) -> media=0.8207  desv.est.=0.0039
  Matriz de confusión: [[1002, 172], [229, 828]]

=== Naïve Bayes ===
  Entrenamiento (accuracy): 0.7412
  Prueba  -> accuracy=0.7481  precision=0.8331  recall=0.5856  f1=0.6878  auc=0.8447
  Validación cruzada (5) -> media=0.7387  desv.est.=0.0021
  Matriz de confusión: [[1050, 124], [438, 619]]

=== SVM ===
  Entrenamiento (accuracy): 0.8765
  Prueba  -> accuracy=0.8422  precision=0.8279  recall=0.8420  f1=0.8349  auc=0.9155
  Validación cruzada (5) -> media=0.8443  desv.est.=0.0037
  Matriz 

### 3.3. Tabla comparativa de métricas

In [ ]:
tabla = pd.DataFrame(resultados)
tabla = tabla.sort_values('accuracy', ascending=False)
tabla_fmt = tabla[['modelo','accuracy','precision','recall','f1','auc','cv_mean','cv_std']].round(4)
tabla_fmt.columns = ['Modelo','Accuracy','Precision','Recall','F1','AUC','CV_media','CV_std']
print(tabla_fmt.to_string(index=False))

                    Modelo  Accuracy  Precision  Recall      F1     AUC   CV_media  CV_std
                       SVM    0.8422     0.8279  0.8420  0.8349  0.9155     0.8443  0.0037
       Regresión Logística    0.8203     0.8280  0.7833  0.8051  0.9071     0.8207  0.0039
   Árbol de Decisión (CART)    0.7853     0.7806  0.7606  0.7705  0.7841     0.7833  0.0104
               Naïve Bayes    0.7481     0.8331  0.5856  0.6878  0.8447     0.7387  0.0021


### 3.4. Análisis del sobreajuste (objetivo específico 3)

El **Árbol de Decisión (CART)** alcanza un **100 % de accuracy en entrenamiento** frente a solo **78,33 %** en validación cruzada (brecha de +21,67 puntos): evidencia clara de **sobreajuste (overfitting)**, pues el modelo memoriza los datos de entrenamiento pero no generaliza.

Tras aplicar la **validación cruzada** como mecanismo de control, el modelo con mejor rendimiento **generalizable** es el **SVM**, con una exactitud (accuracy) de **84,43 %** en validación cruzada (desviación estándar de solo 0,0037, lo que indica gran estabilidad entre pliegues). El SVM presenta además el mayor AUC (0,9155), confirmándolo como el mejor clasificador.

In [ ]:
print('Brecha entrenamiento vs validación cruzada (indicador de sobreajuste):')
for r in resultados:
    gap = r['accuracy_train'] - r['cv_mean']
    print(f"  {r['modelo']:<26} train={r['accuracy_train']:.4f}  cv={r['cv_mean']:.4f}  brecha={gap:+.4f}")

Brecha entrenamiento vs validación cruzada (indicador de sobreajuste):
  Árbol de Decisión (CART)   train=1.0000  cv=0.7833  brecha=+0.2167
  Regresión Logística        train=0.8213  cv=0.8207  brecha=+0.0006
  Naïve Bayes                train=0.7412  cv=0.7387  brecha=+0.0025
  SVM                        train=0.8765  cv=0.8443  brecha=+0.0322


### 3.5. Información de reproducibilidad

In [ ]:
import sklearn, sys
print('Python      :', sys.version.split()[0])
print('scikit-learn:', sklearn.__version__)
print('Semilla (random_state):', RANDOM_STATE)
print('Validación cruzada: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)')

Python      : 3.12.3
scikit-learn: 1.8.0
Semilla (random_state): 42
Validación cruzada: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
